# Logistic Regression — IEEE-CIS Fraud Detection

MLflow experiment: `LogisticRegression_Training`

წრფივი baseline მოდელი. ვნახოთ რამდენს გვაძლევს regularization-ის ცვლილება.

## Setup

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")

# Kaggle-ზე uncomment-ი:
# from kaggle_secrets import UserSecretsClient
# s = UserSecretsClient()
# os.environ["DAGSHUB_TOKEN"]    = s.get_secret("DAGSHUB_TOKEN")
# os.environ["DAGSHUB_USERNAME"] = s.get_secret("DAGSHUB_USERNAME")
# os.environ["DAGSHUB_REPO"]     = s.get_secret("DAGSHUB_REPO")

import dagshub, mlflow
dagshub.init(
    repo_owner=os.environ["DAGSHUB_USERNAME"],
    repo_name=os.environ["DAGSHUB_REPO"],
    mlflow=True,
)
os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ["DAGSHUB_USERNAME"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["DAGSHUB_TOKEN"]

mlflow.set_experiment("LogisticRegression_Training")

## Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

KAGGLE = "/kaggle/input/ieee-fraud-detection"
LOCAL  = "./data"
PATH   = KAGGLE if os.path.exists(KAGGLE) else LOCAL

tx = pd.read_csv(f"{PATH}/train_transaction.csv")
id_ = pd.read_csv(f"{PATH}/train_identity.csv")
df = tx.merge(id_, on="TransactionID", how="left")
del tx, id_

print("shape:", df.shape)
print("fraud rate:", round(df["isFraud"].mean(), 4))


## Cleaning

ჯერ ვაცლებ სვეტებს, სადაც missing ძალიან ბევრია. duplicate row-ებსაც ვშლი.

In [ ]:
with mlflow.start_run(run_name="LogisticRegression_Cleaning"):
    # ვნახოთ რომელ სვეტებს აქვს ყველაზე მეტი missing
    null_pct = df.isna().mean()
    drop_cols = null_pct[null_pct > 0.95].index.tolist()
    df = df.drop(columns=drop_cols)

    n_dup = df.duplicated().sum()
    df = df.drop_duplicates().reset_index(drop=True)

    mlflow.log_params({
        "dropped_cols": len(drop_cols),
        "null_threshold": 0.95,
        "duplicates_removed": int(n_dup),
    })
    mlflow.log_metrics({
        "rows": len(df),
        "cols": df.shape[1],
        "fraud_rate": float(df["isFraud"].mean()),
    })
    print(f"Cleaning-ის შემდეგ: {df.shape}, fraud rate: {df['isFraud'].mean():.4f}")


## Feature Engineering

log-transform ვცადე TransactionAmt-ზე — skewed-ია. D სვეტებიდან mean-ი ავიღე, M სვეტებში T/F-ების დათვლა.

In [ ]:
with mlflow.start_run(run_name="LogisticRegression_Feature_Engineering"):
    df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])

    d_cols = [c for c in df.columns if c.startswith("D") and c[1:].isdigit()]
    if d_cols:
        df["D_mean"] = df[d_cols].mean(axis=1)

    m_cols = [c for c in df.columns if c.startswith("M") and c[1:].isdigit()]
    if m_cols:
        df["M_T_count"] = (df[m_cols] == "T").sum(axis=1)
        df["M_F_count"] = (df[m_cols] == "F").sum(axis=1)

    c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
    if c_cols:
        df["C_sum"] = df[c_cols].fillna(0).sum(axis=1)

    mlflow.log_params({"new_features": "TransactionAmt_log, D_mean, M_T_count, M_F_count, C_sum"})
    mlflow.log_metrics({"total_cols": df.shape[1]})
    print("Feature Engineering დასრულდა. სვეტები:", df.shape[1])


## Feature Selection

Logistic Regression-ისთვის ვარდნა ბევრი სვეტის გამო სლო. VarianceThreshold ვცადე, მერე SelectKBest f_classif. საბოლოოდ კოლინეარობის ამოშლა.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

y = df["isFraud"]
X = df.drop(columns=["isFraud", "TransactionID"], errors="ignore")

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

with mlflow.start_run(run_name="LogisticRegression_Feature_Selection"):
    # VarianceThreshold-ის ანალოგი — მაინც ვტოვებ ყველა სვეტს LR-ისთვის
    # SelectKBest — top 60 სვეტი f_classif-ით
    from sklearn.preprocessing import StandardScaler
    from sklearn.impute import SimpleImputer
    import scipy.stats as stats

    # simple test: correlation with target
    corr = X[num_cols].fillna(-999).corrwith(y).abs().sort_values(ascending=False)
    top_num = corr.head(50).index.tolist()
    selected = top_num + cat_cols
    X = X[selected]
    num_cols = [c for c in num_cols if c in selected]

    mlflow.log_params({
        "method": "correlation_top50_num + all_cat",
        "n_before": len(num_cols) + len(cat_cols),
        "n_after": len(selected),
    })
    mlflow.log_metrics({"features_selected": len(selected)})
    print("Selected features:", len(selected))


## Training

### Underfit — C=0.001 (ძალიან strong regularization)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import tempfile, pathlib

numeric_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
])
cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])
prep = ColumnTransformer([("num", numeric_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

with mlflow.start_run(run_name="LogisticRegression_Underfit"):
    model = LogisticRegression(C=0.001, max_iter=300, solver="saga",
                               class_weight="balanced", random_state=42)
    pipe = Pipeline([("prep", prep), ("clf", model)])
    pipe.fit(X_tr, y_tr)

    tr_auc = roc_auc_score(y_tr, pipe.predict_proba(X_tr)[:,1])
    te_auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:,1])
    mlflow.log_params({"C": 0.001, "note": "underfit - too much regularization"})
    mlflow.log_metrics({"train_auc": tr_auc, "test_auc": te_auc, "gap": tr_auc - te_auc})
    print(f"Underfit  train={tr_auc:.4f}  test={te_auc:.4f}  gap={tr_auc-te_auc:.4f}")


### Overfit — C=100 (თითქმის არანაირი regularization)

In [ ]:
with mlflow.start_run(run_name="LogisticRegression_Overfit"):
    model2 = LogisticRegression(C=100, max_iter=300, solver="saga",
                                class_weight="balanced", random_state=42)
    pipe2 = Pipeline([("prep", prep), ("clf", model2)])
    pipe2.fit(X_tr, y_tr)

    tr_auc2 = roc_auc_score(y_tr, pipe2.predict_proba(X_tr)[:,1])
    te_auc2 = roc_auc_score(y_te, pipe2.predict_proba(X_te)[:,1])
    mlflow.log_params({"C": 100, "note": "overfit risk - no regularization"})
    mlflow.log_metrics({"train_auc": tr_auc2, "test_auc": te_auc2, "gap": tr_auc2 - te_auc2})
    print(f"Overfit   train={tr_auc2:.4f}  test={te_auc2:.4f}  gap={tr_auc2-te_auc2:.4f}")


### Best Model — C=0.1

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, classification_report, average_precision_score
from mlflow.models.signature import infer_signature

with mlflow.start_run(run_name="LogisticRegression_BestModel") as run:
    best = LogisticRegression(C=0.1, max_iter=500, solver="saga",
                              class_weight="balanced", random_state=42)
    best_pipe = Pipeline([("prep", prep), ("clf", best)])
    best_pipe.fit(X_tr, y_tr)

    y_pred  = best_pipe.predict(X_te)
    y_score = best_pipe.predict_proba(X_te)[:,1]

    mlflow.log_params({"C": 0.1, "solver": "saga", "class_weight": "balanced"})
    mlflow.log_metrics({
        "test_auc": float(roc_auc_score(y_te, y_score)),
        "train_auc": float(roc_auc_score(y_tr, best_pipe.predict_proba(X_tr)[:,1])),
        "f1":  float(f1_score(y_te, y_pred, zero_division=0)),
        "precision": float(precision_score(y_te, y_pred, zero_division=0)),
        "recall": float(recall_score(y_te, y_pred, zero_division=0)),
    })

    with tempfile.TemporaryDirectory() as tmp:
        tmp = pathlib.Path(tmp)
        cm = confusion_matrix(y_te, y_pred)
        fig, ax = plt.subplots(figsize=(4,3))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Legit","Fraud"], yticklabels=["Legit","Fraud"])
        ax.set_title("Confusion Matrix")
        fig.tight_layout(); fig.savefig(tmp/"confusion_matrix.png", dpi=100); plt.close()
        (tmp/"report.txt").write_text(
            classification_report(y_te, y_pred, target_names=["Legit","Fraud"], zero_division=0))
        mlflow.log_artifacts(str(tmp))

    sig = infer_signature(X_tr, best_pipe.predict(X_tr))
    mlflow.sklearn.log_model(best_pipe, artifact_path="model", signature=sig)
    print("BestModel run_id:", run.info.run_id)


## დასკვნა

Logistic Regression-ი კარგი baseline-ია, მაგრამ non-linear pattern-ებს ვერ ჭერს. C=0.001 ძალიან underfit-ს. C=100 risk-ს ქმნის. C=0.1 ბალანსზეა.